In [2]:
from pathlib import Path
import hashlib, json, os, platform, subprocess, sys, zipfile
import psutil

assert Path('/kaggle/working').is_dir(), 'Run this notebook on Kaggle CPU'
assert sys.version_info[:2] == (3, 13), 'This offline wheel bundle targets Python 3.13'
COMMIT = '859df1345a6d5034d2202823232f219f34f87ab4'
manifest_files = list(Path('/kaggle/input').rglob('bundle-info.json'))
if not manifest_files:
    archives = list(Path('/kaggle/input').rglob('matchmate-kaggle-offline.zip'))
    assert len(archives) == 1, 'Attach the MatchMate offline input first'
    extracted = Path('/kaggle/working/matchmate-input')
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(extracted)
    manifest_files = [extracted / 'bundle-info.json']
assert len(manifest_files) == 1
INPUT = manifest_files[0].parent
manifest = json.loads(manifest_files[0].read_text())
assert manifest['source_commit'] == COMMIT
for name, expected in manifest['files'].items():
    assert hashlib.sha256((INPUT / name).read_bytes()).hexdigest() == expected, name
PROJECT = Path('/kaggle/working/matchmate-backend')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--quiet', '--branch', 'main', str(INPUT / 'matchmate-source.bundle'), str(PROJECT)], check=True)
subprocess.run(['git', 'checkout', '--quiet', '--detach', COMMIT], cwd=PROJECT, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip() == COMMIT
os.chdir(PROJECT)
VENV = Path('/kaggle/working/matchmate-venv')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', '--no-index', '--find-links', str(INPUT / 'wheels'), '-r', 'requirements-load.txt'], check=True)
environment = {'source_commit': COMMIT, 'python': sys.version, 'platform': platform.platform(), 'cpu_count': os.cpu_count(), 'ram_bytes': psutil.virtual_memory().total, 'kernel_psutil': psutil.__version__, 'internet': 'off', 'accelerator': 'none'}
Path('/kaggle/working/kaggle-environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))
subprocess.run([PYTHON, '-m', 'ruff', 'check', '.'], check=True)
subprocess.run([PYTHON, '-m', 'ruff', 'format', '--check', '.'], check=True)
subprocess.run([PYTHON, '-m', 'pytest', '-q'], check=True)
print('READY: pinned source, offline dependencies and functional tests passed')


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Looking in links: /kaggle/input/datasets/datdooo/matchmate-offline-load-test-859df/wheels
Processing /kaggle/input/datasets/datdooo/matchmate-offline-load-test-859df/wheels/fastapi-0.143.0-py3-none-any.whl (from -r /kaggle/working/matchmate-backend/requirements.txt (line 2))
Processing /kaggle/input/datasets/datdooo/matchmate-offline-load-test-859df/wheels/uvicorn-0.54.0-py3-none-any.whl (from -r /kaggle/working/matchmate-backend/requirements.txt (line 3))
Processing /kaggle/input/datasets/datdooo/matchmate-offline-load-test-859df/wheels/email_validator-2.3.0-py3-none-any.whl (from -r /kaggle/working/matchmate-backend/requirements.txt (line 4))
Processing /kaggle/input/datasets/datdooo/matchmate-offline-load-test-859df/wheels/sqlalchemy-2.1.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (from -r /kaggle/working/matchmate-backend/requirements.txt (line 6))
Processing /kaggle/input/datasets/datdooo/matchmate-offline-load-test-859df/wheels/pyjwt-2.15.1-

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


All checks passed!
93 files already formatted
........................................................................ [ 30%]
........................................................................ [ 61%]
........................................................................ [ 91%]
...................                                                      [100%]
=============================== warnings summary ===============================
../matchmate-venv/lib/python3.13/site-packages/starlette/testclient.py:45
  /kaggle/working/matchmate-venv/lib/python3.13/site-packages/starlette/testclient.py:45: DeprecationWarning: The anyio.abc.BlockingPortal alias is deprecated, use anyio.from_thread.BlockingPortal instead.
    _PortalFactoryType = Callable[[], AbstractContextManager[anyio.abc.BlockingPortal]]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
235 passed, 1 warning in 36.57s
READY: pinned source, offline dependencies and functional tests passed


In [3]:
from pathlib import Path
import json, shutil, subprocess, zipfile

result = subprocess.run([PYTHON, '-m', 'scripts.run_kaggle_load_test', '--profile', 'full'], cwd=PROJECT)
ARTIFACTS = PROJECT / 'artifacts'
shutil.copy2('/kaggle/working/kaggle-environment.json', ARTIFACTS / 'kaggle-environment.json')
shutil.copy2(INPUT / 'bundle-info.json', ARTIFACTS / 'offline-bundle-info.json')
records = [json.loads(path.read_text()) for path in sorted(ARTIFACTS.glob('locust-*-metadata.json'))]
summary = {'source_commit': COMMIT, 'runner_exit_code': result.returncode, 'case_count': len(records), 'cases': [{key: row.get(key) for key in ('profile', 'virtual_users', 'environment', 'logged_in_users', 'measurement_complete', 'exit_code', 'errors', 'invariants', 'aggregate')} for row in records]}
(ARTIFACTS / 'run-summary.json').write_text(json.dumps(summary, indent=2))
archive_path = Path('/kaggle/working/matchmate-kaggle-results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ARTIFACTS.rglob('*')):
        if path.is_file():
            archive.write(path, Path('artifacts') / path.relative_to(ARTIFACTS))
    archive.write(PROJECT / 'docs/LOAD_TEST_RESULTS.md', 'LOAD_TEST_RESULTS.md')
shutil.copy2(PROJECT / 'docs/LOAD_TEST_RESULTS.md', '/kaggle/working/LOAD_TEST_RESULTS.md')
print(json.dumps(summary, indent=2))
print('ARTIFACT_ZIP', archive_path)
print('REPORT', '/kaggle/working/LOAD_TEST_RESULTS.md')
assert len(records) == 6 and all(row['environment'] == 'kaggle' and row['measurement_complete'] for row in records), 'Incomplete measurement; inspect saved logs and metadata'
assert not any(any(row['invariants'].values()) for row in records), 'Invariant violation; inspect saved artifacts'
if result.returncode:
    raise RuntimeError('Load runner reported failures; artifacts were saved for review')


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Chạy read: 10 virtual users
Chạy read: 50 virtual users
Chạy read: 100 virtual users
Chạy lifecycle: 10 virtual users
Chạy lifecycle: 50 virtual users
Chạy lifecycle: 100 virtual users
{
  "source_commit": "859df1345a6d5034d2202823232f219f34f87ab4",
  "runner_exit_code": 1,
  "case_count": 6,
  "cases": [
    {
      "profile": "lifecycle",
      "virtual_users": 100,
      "environment": "kaggle",
      "logged_in_users": 42,
      "measurement_complete": false,
      "exit_code": 1,
      "errors": {},
      "invariants": {
        "over_capacity": 0,
        "accepted_overlap": 0,
        "active_in_cancelled": 0,
        "open_without_host": 0
      },
      "aggregate": {}
    },
    {
      "profile": "lifecycle",
      "virtual_users": 10,
      "environment": "kaggle",
      "logged_in_users": 10,
      "measurement_complete": true,
      "exit_code": 0,
      "errors": {},
      "invariants": {
        "over_capacity": 0,
        "accepted_overlap": 0,
        "active_in_cance

AssertionError: Incomplete measurement; inspect saved logs and metadata